# YouTube Video Analytics – Compact & Error-Free Python Project

**Goal:** Clean the YouTube analytics dataset, analyze category, traffic, device, and country performance, generate 3 key charts, and export the cleaned data without any file/permission errors.

## Step 1 – Import Libraries & Safe Data Load

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# Safe output folder in Downloads to avoid Windows PermissionError
base_dir = os.path.join(os.path.expanduser("~"), "Downloads", "YouTube_Analytics_Output")
os.makedirs(base_dir, exist_ok=True)

# Auto-locate Excel dataset
file_name = "YouTube_Video_Analytics_Dataset.xlsx"
home = os.path.expanduser("~")
possible_paths = [
    file_name,
    os.path.join(home, "Downloads", file_name),
    os.path.join(home, "Desktop", file_name),
    f"../data/raw/{file_name}",
    f"data/raw/{file_name}"
]
file_path = next((p for p in possible_paths if os.path.exists(p)), None)

if file_path is None:
    raise FileNotFoundError(f"'{file_name}' not found! Please place the file in your Downloads, Desktop, or current folder.")

raw = pd.read_excel(file_path, sheet_name="YouTube_Video_Data")
print(f"Dataset loaded successfully from: {file_path}")
print("Shape:", raw.shape)

## Step 2 – Data Cleaning & Feature Engineering

In [ ]:
# 1. Remove summary row (missing Video Title) and clean text columns
df = raw.dropna(subset=["Video Title"]).copy()
for col in ["Video Title", "Category", "Top Traffic Source", "Top Device Type", "Top Country"]:
    df[col] = df[col].astype(str).str.strip()

df["Publish Date"] = pd.to_datetime(df["Publish Date"], errors="coerce")

# 2. Set zero RPM/Revenue to missing (NaN)
df.loc[df["Estimated Revenue USD"] == 0, ["RPM USD", "Estimated Revenue USD"]] = np.nan

# 3. Rename columns for convenience
df = df.rename(columns={
    "Video Title": "Title", "Publish Date": "Publish_Date", "Duration Sec": "Duration_Sec",
    "Impressions CTR %": "CTR_Pct", "Watch Time Hours": "Watch_Hours",
    "Avg View Duration Sec": "Avg_View_Sec", "Avg Percentage Viewed %": "Avg_Pct_Viewed",
    "Subscribers Gained": "Subs_Gained", "Engagement Rate %": "Engagement_Pct",
    "Top Traffic Source": "Traffic_Source", "Top Device Type": "Device",
    "Top Country": "Country", "RPM USD": "RPM_USD", "Estimated Revenue USD": "Revenue_USD"
}).reset_index(drop=True)

# 4. Create derived analysis columns
df["Duration_Min"] = df["Duration_Sec"] / 60
df["Is_Series"] = df["Title"].str.contains(r"\(Ep\.", regex=True)
df["Revenue_Per_1000_Views"] = (df["Revenue_USD"] / df["Views"]) * 1000

print("Clean dataset shape:", df.shape)
display(df.head(3))

## Step 3 – Summary Tables (Category, Traffic, Device, Country)

In [ ]:
cat = (df.groupby("Category")
         .agg(Videos=("Title", "count"), Total_Views=("Views", "sum"),
              Total_Revenue=("Revenue_USD", "sum"), Rev_per_1000=("Revenue_Per_1000_Views", "median"),
              Avg_Engagement=("Engagement_Pct", "mean"))
         .sort_values("Total_Revenue", ascending=False))

def get_summary(col):
    return (df.groupby(col)
              .agg(Videos=("Title", "count"), Total_Views=("Views", "sum"), Total_Revenue=("Revenue_USD", "sum"))
              .sort_values("Total_Views", ascending=False))

src, dev, cty = get_summary("Traffic_Source"), get_summary("Device"), get_summary("Country")
display(cat)

## Step 4 – Visualizations (3 Key Graphs)

In [ ]:
# Graph 1: Category Performance
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
order = cat.sort_values("Total_Views", ascending=False).index

sns.barplot(x=cat.loc[order, "Total_Views"], y=order, ax=ax[0], color="#3b78b5")
ax[0].set_title("Total Views by Category"); ax[0].set_ylabel("")

sns.barplot(x=cat.loc[order, "Total_Revenue"] / 1e6, y=order, ax=ax[1], color="#e07b39")
ax[1].set_title("Total Revenue (USD Millions)"); ax[1].set_ylabel("")

sns.barplot(x=cat.loc[order, "Rev_per_1000"] / 1e3, y=order, ax=ax[2], color="#4c9a6a")
ax[2].set_title("Median Rev / 1,000 Views ($K)"); ax[2].set_ylabel("")

plt.tight_layout()
plt.savefig(os.path.join(base_dir, "01_category_performance.png"), bbox_inches="tight")
plt.show()

# Graph 2: Share of Views by Traffic Source, Device & Country
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
for a, (title, t) in zip(ax, [("Traffic Source", src), ("Device", dev), ("Country", cty)]):
    share = (t["Total_Views"] / t["Total_Views"].sum()) * 100
    sns.barplot(x=share.values, y=share.index, ax=a, color="#7a5aa6")
    a.set_title(f"Views Share by {title}"); a.set_xlabel("% of Total Views"); a.set_ylabel("")
    for i, v in enumerate(share.values):
        a.text(v + 0.5, i, f"{v:.0f}%", va="center", fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(base_dir, "02_traffic_device_country.png"), bbox_inches="tight")
plt.show()

# Graph 3: Correlation Heatmap
num_cols = ["Duration_Sec", "Impressions", "CTR_Pct", "Views", "Watch_Hours",
            "Avg_Pct_Viewed", "Likes", "Comments", "Shares", "Subs_Gained", "Engagement_Pct", "Revenue_USD"]
plt.figure(figsize=(10, 7))
sns.heatmap(df[num_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, linewidths=0.5)
plt.title("Correlation Matrix of Video Metrics")
plt.tight_layout()
plt.savefig(os.path.join(base_dir, "03_correlation_heatmap.png"), bbox_inches="tight")
plt.show()

## Step 5 – Export Clean Files & Key Insights

In [ ]:
df.to_csv(os.path.join(base_dir, "youtube_video_clean.csv"), index=False)
cat.to_csv(os.path.join(base_dir, "summary_by_category.csv"))

print("1. Top Revenue & Views Category :", cat["Total_Revenue"].idxmax())
print("2. Top Traffic / Device / Country:", f"{src.index[0]} / {dev.index[0]} / {cty.index[0]}")
print("3. Duration vs Views Correlation :", round(df["Duration_Min"].corr(df["Views"]), 3))
print(f"\nAll cleaned CSVs and 3 charts saved safely in:\n{base_dir}")